# Consensus at a Turning Point: Analyzing Ethereum's 2022 Migration From PoW to PoS
### End-to-End Computational Research Notebook

This notebook reproduces the empirical findings and computational models of the thesis:
1. **Data Ingestion & Preprocessing**
2. **Exploratory Data Analysis (EDA)**
3. **Correlation Analysis (Hash Rate, Transactions, Energy)**
4. **ARIMA(5,1,2) Time-Series Forecasting & Validation**
5. **Hypothetical Counterfactual Continued-PoW Scenario**
6. **PoW vs PoS Sustainability Assessment**

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from src.utils.data_loader import load_ethereum, load_bitcoin
from src.analysis.correlation.correlation import eth_pow_correlations, summarise_correlations
from src.analysis.trends.trends import compute_pow_pos_change, split_pre_post_merge
from src.analysis.forecasting.arima_model import run_forecast_pipeline, run_counterfactual_pow
from src.analysis.eda.visualizations import plot_metric_over_time, plot_pow_pos_comparison, plot_arima_forecast

df_eth = load_ethereum()
df_btc = load_bitcoin()
print(f"Ethereum rows: {len(df_eth)} | Bitcoin rows: {len(df_btc)}")

## 1. Ethereum PoW Correlation Analysis
Evaluating thesis findings:
- Hash Rate vs Energy: r ≈ 0.96
- Transaction Count vs Hash Rate: r ≈ 0.18
- Transaction Count vs Energy: r ≈ 0.11

In [ ]:
corrs = eth_pow_correlations(df_eth)
corr_summary = summarise_correlations(corrs)
corr_summary[corr_summary["method"] == "Pearson"][["x", "y", "r", "p_value", "significant"]]

## 2. Pre-Merge vs Post-Merge Structural Changes

In [ ]:
change_df = compute_pow_pos_change(df_eth)
change_df

## 3. ARIMA(5,1,2) Modeling & Holdout Evaluation

In [ ]:
arima_res = run_forecast_pipeline(
    df=df_eth,
    metric="transaction_count",
    blockchain="ethereum",
    test_size=0.15,
    forecast_horizon=90,
    order=(5, 1, 2)
)
print(f"MAE: {arima_res.mae:.2f} | RMSE: {arima_res.rmse:.2f} | MAPE: {arima_res.mape:.2f}%")

## 4. Counterfactual Continued-PoW Simulation
Simulating what Ethereum's hash rate would have looked like had it continued using PoW after Sep 2022.

In [ ]:
cf_res = run_counterfactual_pow(df_eth, metric="hash_rate", forecast_horizon=365, order=(5, 1, 2))
cf_res.forecast_df.head(10)